<h1 style="color: red; text-align:center">Model Training</br>

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

import optuna
from tqdm import tqdm

c:\Users\posze\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [3]:
from src.data import build_dataloaders
from src.feature_extraction_1 import FeatureExtractor
from src.parser import parse_gzip, filter_proteins
from src.encoder import ProteinEncoder


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
train_loader, val_loader, test_loader, mlb, norm_stats = build_dataloaders(
    file_path='../data/uniprot_sprot.dat.gz',
    max_sequence_length=1000,
    min_go_term_count=100,
    batch_size=64,
    val_split=0.15,
    test_split=0.15,
    num_workers=4
)
    

Parsing proteins...
535,892 proteins after length/annotation filter
531,882 proteins | 3,506 GO terms (min_count=100)
Split: train=372,316 | val=79,783 | test=79,783


In [6]:
class BioNet(nn.Module):
    """
    Simple MLP to process the hand-crafted features
    """
    def __init__(self, input_dim: int, output_dim:int):
        super(BioNet, self).__init__()
        
        self.block = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self,X):
        return self.block(X)

In [7]:
class ProteinFunctionPredictor(nn.Module):
    """
    Combines the hand-crafted features with a simple MLP to 
    predict the GO terms
    """

    def __init__(self,
                 physics_dim: int,
                 num_go_terms: int,
                 cnn_channels: int = 64,
                 cnn_depth: int = 3,
                 physics_out: int = 64,
                 dropout: float = 0.3):
        super(ProteinFunctionPredictor, self).__init__()
        
        self.bionet = BioNet(
            input_dim =physics_dim,
            output_dim = physics_out
        )

        self.protein_encoder = ProteinEncoder(
            in_channels = 20, 
            branch_channels = cnn_channels, 
            kernel_sizes = [3, 5, 9], 
            depth = cnn_depth,
            dropout = dropout
        )

        combined_dim = physics_out + 2 * cnn_channels * 3 # 2 pools per branch [max, mean]

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_go_terms)
        )

    def forward(self,
                physics_features: torch.Tensor,
                sequence_one_hot: torch.Tensor
                ) -> torch.Tensor:
        """
        Args:
            physics_features: Tensor of shape (batch_size, physics_dim)
            sequence_one_hot: Tensor of shape (batch_size, 20, seq_len)

        Returns:
            Tensor of shape (batch_size, num_go_terms) with raw logits
        """
        physics_out = self.bionet(physics_features) # (batch_size, physics_out)
        sequence_out = self.protein_encoder(sequence_one_hot) # (batch_size, cnn_out)

        combined = torch.cat([physics_out, sequence_out], dim=1) # (batch_size, combined_dim)
        logits = self.classifier(combined)
        return logits # (batch_size, num_go_terms)
    
    def predict_proba(self,
                      physics_features: torch.Tensor,
                      sequence_one_hot: torch.Tensor
                      ) -> torch.Tensor:
        """
        Returns predicted probabilities for each GO term
        """
        logits = self.forward(physics_features, sequence_one_hot)
        return torch.sigmoid(logits)



In [ ]:
def build_best_model(train_loader, val_loader, norm_stats):
    
    skf = StratifiedKFold(n_splits=5, shuffle=True)

    def objective(trial: optuna.Trial):
        batch_size = 64
        lr = trial.suggest_float('lr', 1e-5, 0.01, log=True)
        dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
        cnn_channels = trial.suggest_categorical('cnn_channels', [32, 64, 128])
        cnn_depth = trial.suggest_int('cnn_depth', 2, 4)

        all_sequence_one_hot = []
        all_physics_features = []
        all_labels = []
        best_val_losses = [np.inf] * skf.n_splits

        for sequence_one_hot, physics_features, labels in train_loader:
            all_sequence_one_hot.append(sequence_one_hot.cpu())
            all_physics_features.append(physics_features.cpu())
            all_labels.append(labels.cpu())

        for sequence_one_hot, physics_features, labels in val_loader:
            all_sequence_one_hot.append(sequence_one_hot.cpu())
            all_physics_features.append(physics_features.cpu())
            all_labels.append(labels.cpu())
        
        all_sequence_one_hot = torch.cat(all_sequence_one_hot, dim=0)
        all_physics_features = torch.cat(all_physics_features, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        fold_val_losses = np.zeros(skf.n_splits)

        for fold, (train_idx, val_idx) in enumerate(skf.split(all_physics_features, all_labels)):
            print(f"Fold {fold+1}/{skf.n_splits}")
            fold_train_loader = DataLoader(
                torch.utils.data.TensorDataset(all_sequence_one_hot[train_idx], all_physics_features[train_idx], all_labels[train_idx]),
                batch_size=batch_size,
                shuffle=True
            )
            fold_val_loader = DataLoader(
                torch.utils.data.TensorDataset(all_sequence_one_hot[val_idx], all_physics_features[val_idx], all_labels[val_idx]),
                batch_size=batch_size,
                shuffle=False
            )

            model = ProteinFunctionPredictor(
                physics_dim=norm_stats['physics_dim'],
                num_go_terms=norm_stats['num_go_terms'],
                cnn_channels=cnn_channels,
                cnn_depth=cnn_depth,
                dropout=dropout
            ).to(DEVICE)

            loss_fn = nn.BCEWithLogitsLoss()
            optimizer = optim.Adam(model.parameters(), lr=lr)
            num_epochs = 10
            best_val_loss = float('inf')
            train_losses = []
            val_losses = []
            
            for epoch in range(num_epochs):
                model.train()
                train_loss = 0.0    

                for sequence_one_hot, physics_features, labels in tqdm(fold_train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
                    sequence_one_hot = sequence_one_hot.to(DEVICE)
                    physics_features = physics_features.to(DEVICE)
                    labels = labels.to(DEVICE)

                    optimizer.zero_grad()
                    yhat = model(physics_features, sequence_one_hot)
                    loss = loss_fn(yhat, labels)
                    
                    loss.backward()
                    optimizer.step()

                    train_loss += loss.item() * physics_features.size(0)
            
                train_loss /= len(fold_train_loader.dataset)
                train_losses.append(train_loss)

                with torch.inference_mode():
                    model.eval()
                    val_loss = 0.0

                    for sequence_one_hot, physics_features, labels in tqdm(fold_val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
                        sequence_one_hot = sequence_one_hot.to(DEVICE)
                        physics_features = physics_features.to(DEVICE)
                        labels = labels.to(DEVICE)

                        yhat = model(physics_features, sequence_one_hot)
                        loss = loss_fn(yhat, labels)
                        val_loss += loss.item() * physics_features.size(0)
                
                    val_loss /= len(fold_val_loader.dataset)
                    val_losses.append(val_loss)

                    if val_loss < best_val_loss and val_loss < best_val_losses[fold]:
                        best_val_loss = val_loss
                        best_val_losses[fold] = val_loss
                        dic = {
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'hyperparameters': {
                                'batch_size': batch_size,
                                'lr': lr,
                                'dropout': dropout,
                                'cnn_channels': cnn_channels,
                                'cnn_depth': cnn_depth
                            },
                            'train_losses': train_losses,
                            'val_losses': val_losses
                        }

                        torch.save(dic, f'../src/models/best_model_fold_{fold}.pth')

                trial.report(val_loss, epoch)

                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            fold_val_losses[fold] = best_val_loss   

        return np.mean(fold_val_losses)
    
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=20)

    final_models = []
    for fold in range(5):
        final_models.append(ProteinFunctionPredictor(
            physics_dim=norm_stats['physics_dim'],
            num_go_terms=norm_stats['num_go_terms'],
            cnn_channels=study.best_params['cnn_channels'],
            cnn_depth=study.best_params['cnn_depth'],
            dropout=study.best_params['dropout']
        ).to(DEVICE))

        checkpoint = torch.load(f'../src/models/best_model_fold_{fold}.pth')
        final_models[fold].load_state_dict(checkpoint['model_state_dict'])

    return final_models, study.best_params
   

In [ ]:
models, best_params = build_best_model(train_loader, val_loader, norm_stats)
print("Best Hyperparameters:", best_params)

[I 2026-04-27 17:24:01,457] A new study created in memory with name: no-name-dfd9605d-a220-4092-9fce-232b8086e8a8
Epoch 10/10 - Validation: 100%|██████████| 1247/1247 [00:58<00:00, 21.15it/s]
[I 2026-04-27 18:42:22,388] Trial 0 finished with value: 0.0034865101468737262 and parameters: {'lr': 0.0013289115910182126, 'dropout': 0.2, 'cnn_channels': 32, 'cnn_depth': 3}. Best is trial 0 with value: 0.0034865101468737262.
Epoch 10/10 - Validation: 100%|██████████| 1247/1247 [00:55<00:00, 22.29it/s]
[I 2026-04-27 19:43:22,223] Trial 1 finished with value: 0.004736235217197081 and parameters: {'lr': 0.005219644365653151, 'dropout': 0.30000000000000004, 'cnn_channels': 32, 'cnn_depth': 2}. Best is trial 0 with value: 0.0034865101468737262.
Epoch 4/10 - Training:   3%|▎         | 155/5818 [01:01<22:40,  4.16it/s] 